# CHAOSS Election Voting Eligibility 


General election information can be found https://github.com/chaoss/community/blob/main/elections/mechanics-GB.md
One of the options for creating voting criteria is a contribtion in the last year. This notebook will gather the Github logins for the individuals who have a contribution in either the github.com/chaoss
or github.com/badging orgs in the past year

In [ ]:
import psycopg2
import pandas as pd 
import sqlalchemy as salc
import json
import datetime as dt
from datetime import datetime, timedelta, timezone
import numpy as np
import warnings
import plotly
import plotly.express as px
from dateutil.relativedelta import *
import plotly.graph_objects as go
import time
#from fuzzywuzzy import fuzz
warnings.filterwarnings("ignore")
plotly.offline.init_notebook_mode()

with open("../ec2_creds.json") as config_file:
    config = json.load(config_file)

In [2]:
database_connection_string = 'postgresql+psycopg2://{}:{}@{}:{}/{}'.format(config['user'], config['password'], config['host'], config['port'], config['database'])

dbschema='augur_data'
engine = salc.create_engine(
    database_connection_string,
    connect_args={'options': '-csearch_path={}'.format(dbschema)})

This will allow you to get specific repo_ids. A few are listed for ease of use

In [3]:
org_name = 'chaoss'

repo_query = salc.sql.text(f"""
        SELECT 
            r.repo_git,
            r.repo_id,
            r.repo_name,
            rg.rg_name
        FROM
            repo r,
            repo_groups rg
        WHERE 
            rg.repo_group_id = r.repo_group_id AND 
            rg.rg_name = '{org_name}'
        """)

with engine.connect() as connection:
    t = connection.execute(repo_query)
    results = t.all()
    
chaoss_repo_ids = [ row[1] for row in results]
chaoss_repo_names = [ row[0] for row in results]

In [4]:
len(chaoss_repo_ids)

71

In [5]:
chaoss_repo_names

['https://github.com/chaoss/whitepaper',
 'https://github.com/chaoss/grimoirelab-perceval-mozilla',
 'https://github.com/chaoss/grimoirelab-perceval-gitee',
 'https://github.com/chaoss/ai-detection-action',
 'https://github.com/chaoss/wg-ai-alignment',
 'https://github.com/chaoss/unsdg-formsite',
 'https://github.com/chaoss/unsdg-classifier-tool',
 'https://github.com/chaoss/augur-utilities',
 'https://github.com/chaoss/grimoirelab-tutorial',
 'https://github.com/chaoss/grimoirelab-manuscripts',
 'https://github.com/chaoss/afos-api',
 'https://github.com/chaoss/wg-un-sdg',
 'https://github.com/chaoss/chaoss-asia',
 'https://github.com/chaoss/grimoirelab-core',
 'https://github.com/chaoss/hugo-cms',
 'https://github.com/chaoss/wg-app-ecosystem',
 'https://github.com/chaoss/wg-funding-impact',
 'https://github.com/chaoss/mars',
 'https://github.com/chaoss/grimoirelab-chronicler',
 'https://github.com/chaoss/oscdb',
 'https://github.com/chaoss/augur-community-reports',
 'https://github.co

In [6]:
org_name = 'badging'

repo_query = salc.sql.text(f"""
        SELECT 
            r.repo_git,
            r.repo_id,
            r.repo_name,
            rg.rg_name
        FROM
            repo r,
            repo_groups rg
        WHERE 
            rg.repo_group_id = r.repo_group_id AND 
            rg.rg_name = '{org_name}'
        """)

with engine.connect() as connection:
    t = connection.execute(repo_query)
    results = t.all()
    
badging_repo_ids = [ row[1] for row in results]
badging_repo_names = [ row[0] for row in results]

In [7]:
len(badging_repo_ids)

17

In [8]:
repo_ids = chaoss_repo_ids + badging_repo_ids

In [9]:
len(repo_ids)

88

Query for contributions with related contributor information. This query gets the following contributor actions: 
- Commits 
- Issues: open, close, comment 
- Pull Requests: open, close, merge, review, comment

with the associated github login and all associated emails with said github account. 

In [26]:
repo_statement = str(repo_ids)
repo_statement = repo_statement[1:-1]
print(repo_statement)

contrib_query = f"""
SELECT
                        left(c.cntrb_id::text, 15) as cntrb_id, -- first 15 characters of the uuid
                        timezone('utc', c.created_at) AS created_at,
                        c.repo_id,
                        c.login,
                        c.action,
                        con.cntrb_company,
                        array_to_string(
                            ARRAY(
                                SELECT DISTINCT e
                                FROM unnest(
                                    array_agg(ca.alias_email) ||
                                    ARRAY[con.cntrb_email, con.cntrb_canonical]
                                ) AS e
                                WHERE e IS NOT NULL AND e != ''
                            ), ' , '
                        ) AS email_list
                    FROM
                        explorer_contributor_actions c
                    JOIN contributors_aliases ca
                        ON c.cntrb_id = ca.cntrb_id
                    JOIN contributors con
                        ON c.cntrb_id = con.cntrb_id
                    WHERE
    c.repo_id IN ({repo_statement})
GROUP BY
    c.cntrb_id, c.created_at, c.repo_id, c.login, c.action,
    con.cntrb_company, con.cntrb_email, con.cntrb_canonical
                """
df_contributions = pd.read_sql_query(contrib_query, con=engine)

df_contributions = df_contributions.reset_index()
df_contributions.drop("index", axis=1, inplace=True)

df_contributions["created_at"] = pd.to_datetime(df_contributions["created_at"], utc=True)
df_contributions.loc[df_contributions["action"] == "pull_request_review_COMMENTED", "action"] = "PR Review"
df_contributions.loc[df_contributions["action"] == "pull_request_review_APPROVED", "action"] = "PR Review"
df_contributions.loc[df_contributions["action"] == "pull_request_review_CHANGES_REQUESTED", "action"] = "PR Review"
df_contributions.loc[df_contributions["action"] == "pull_request_review_DISMISSED", "action"] = "PR Review"
df_contributions.loc[df_contributions["action"] == "pull_request_open", "action"] = "PR Opened"
df_contributions.loc[df_contributions["action"] == "pull_request_comment", "action"] = "PR Comment"
df_contributions.loc[df_contributions["action"] == "pull_request_closed", "action"] = "PR Closed"
df_contributions.loc[df_contributions["action"] == "pull_request_merged", "action"] = "PR Merged"
df_contributions.loc[df_contributions["action"] == "issue_opened", "action"] = "Issue Opened"
df_contributions.loc[df_contributions["action"] == "issue_closed", "action"] = "Issue Closed"
df_contributions.loc[df_contributions["action"] == "issue_comment", "action"] = "Issue Comment"
df_contributions.loc[df_contributions["action"] == "commit", "action"] = "Commit"
df_contributions.loc[df_contributions["action"] == "pull_request_review_PENDING", "action"] = "PR Review"

25452, 72151, 72187, 104311, 104310, 104305, 104309, 104306, 72150, 72154, 104302, 104303, 104300, 104299, 104312, 72173, 104304, 72182, 104301, 104307, 72177, 72165, 72191, 72155, 72158, 72184, 36113, 104308, 72189, 72175, 72192, 78408, 72174, 72176, 72190, 72159, 72188, 78411, 72163, 72178, 72156, 72147, 72149, 72179, 72181, 72180, 72185, 72145, 72168, 72160, 72169, 72144, 72164, 72166, 72143, 72161, 72148, 72142, 78385, 25450, 78409, 72153, 25445, 78412, 72171, 72172, 78410, 72186, 71441, 72167, 72146, 78687, 78683, 78685, 104293, 104294, 78681, 78686, 78691, 78677, 78680, 78684, 78689, 78679, 78690, 78678, 78682, 78688


In [27]:
df_contributions = df_contributions.sort_values(by='created_at')

In [33]:
df_contributions[df_contributions['login']== 'andrew']

,cntrb_id,created_at,repo_id,login,action,cntrb_company,email_list
3,01000004-2400-0,2025-09-02 19:54:39+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
4,01000004-2400-0,2025-09-17 15:14:32+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
5,01000004-2400-0,2025-09-17 15:14:41+00:00,104308,andrew,PR Merged,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
6,01000004-2400-0,2025-09-17 15:41:27+00:00,104308,andrew,Issue Opened,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
7,01000004-2400-0,2025-09-21 16:34:52+00:00,104308,andrew,PR Opened,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
8,01000004-2400-0,2025-10-15 11:32:32+00:00,104308,andrew,PR Opened,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
9,01000004-2400-0,2025-10-15 14:13:12+00:00,104308,andrew,PR Opened,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
10,01000004-2400-0,2025-10-19 16:00:06+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
11,01000004-2400-0,2025-10-29 13:59:14+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
12,01000004-2400-0,2025-10-29 14:02:48+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"


In [28]:
# 1. Ensure created_at is in datetime format
df_contributions['created_at'] = pd.to_datetime(df_contributions['created_at'])

# 2. Define the cutoff (exactly 1 year ago from right now)
one_year_ago = pd.Timestamp.now(tz='UTC') - pd.DateOffset(years=1)

# 3. Filter the DataFrame
df_contributions_year = df_contributions[df_contributions['created_at'] >= one_year_ago]

In [34]:
df_contributions_year[df_contributions_year['login']== 'andrew']

,cntrb_id,created_at,repo_id,login,action,cntrb_company,email_list
3,01000004-2400-0,2025-09-02 19:54:39+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
4,01000004-2400-0,2025-09-17 15:14:32+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
5,01000004-2400-0,2025-09-17 15:14:41+00:00,104308,andrew,PR Merged,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
6,01000004-2400-0,2025-09-17 15:41:27+00:00,104308,andrew,Issue Opened,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
7,01000004-2400-0,2025-09-21 16:34:52+00:00,104308,andrew,PR Opened,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
8,01000004-2400-0,2025-10-15 11:32:32+00:00,104308,andrew,PR Opened,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
9,01000004-2400-0,2025-10-15 14:13:12+00:00,104308,andrew,PR Opened,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
10,01000004-2400-0,2025-10-19 16:00:06+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
11,01000004-2400-0,2025-10-29 13:59:14+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"
12,01000004-2400-0,2025-10-29 14:02:48+00:00,104308,andrew,PR Review,@ecosyste-ms and @octobox,"andrew.nesbitt@github.com , andrewnez@gmail.com"


In [35]:
unique_login = sorted(df_contributions_year['login'].unique().tolist(), key=str.lower)

In [36]:
unique_login

['1steve78',
 'AbdessamadTzn',
 'ABrain7710',
 'AdeebaNizam404',
 'adeyinkaoresanya',
 'afro-coder',
 'agriyakhetarpal',
 'AJ-droi',
 'alberefe',
 'alextsakpinis',
 'alice-sowerby',
 'aliok',
 'anajsana',
 'andrew',
 'Anita-ihuman',
 'ANJAN672',
 'antcybersec',
 'antoninbas',
 'anusha975',
 'ashutoshsao',
 'atheendre130505',
 'AumOzaa',
 'Aysha-py',
 'bact',
 'badging-bot[bot]',
 'banjoh',
 'bhantsi',
 'blisc',
 'bobidle',
 'bproffitt',
 'Busayo-ojo',
 'canasdiaz',
 'CatherineKiiru',
 'cdolfi',
 'cgwalters',
 'chris-short',
 'cnaples79',
 'CodeBySayak',
 'CoralineAda',
 'daluclemas',
 'DARK-1926',
 'dcermak',
 'dependabot[bot]',
 'DesmondSanctity',
 'Dev10-sys',
 'DhaneshKolu',
 'dipak0000812',
 'divya-mohan0209',
 'djmitche',
 'dtrinf',
 'E-STAT',
 'EjiroOS',
 'ElizabethN',
 'em07-adoz',
 'emmairwin',
 'emphor11',
 'EngCaioFonseca',
 'erikerlandson',
 'evamillan',
 'eyehwan',
 'FaithKovi',
 'fioddor',
 'frontEndDoctor',
 'geekygirldawn',
 'GeorgLink',
 'germonprez',
 'giordano',
 'git

In [37]:
import csv
with open("eligable_voters.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for s in unique_login:
        writer.writerow([s])